In [1]:
import os
import re
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

In [2]:
# =========================================================
# 参数
# =========================================================

input_dir = "/public/users/xueyupeng/VZV/cytof/subtype_combined"

output_root = "/public/users/xueyupeng/VZV/cytof/subtype_marker_analysis"

os.makedirs(output_root, exist_ok=True)

sub_markers = [
    'IgD_142Nd', 'IFN-a_144Nd', 'CXCL9_146Nd', 'CD86_147Sm',
    'H3K27ac_148Nd', 'IL-1b_149Sm', '4E-BP1_150Nd',
    'pS6_151Eu', 'CCR2_152Sm', 'TNF-a_153Eu',
    'IL-4_154Sm', 'CASP-8_156Gd', 'CD38_159Tb',
    'CD123_161Dy', 'Ki-67_162Dy',
    'CXCL10_163Dy', 'Arginase-1_164Dy',
    'IL-6_165Ho', 'MCP-1_166Er',
    'CD69_167Er', 'STC1_168Er',
    'IFN-g_172Yb', 'pSTAT1_175Lu',
    'pSTAT3_176Yb'
]


timepoint_order = [
    "V1Day0",
    "V1Day1",
    "V2Day0",
    "V2Day1"
]

alpha = 0.05


In [3]:
# =========================================================
# 读取函数
# =========================================================

def load_cytof_h5ad(
    file_path,
    do_arcsinh=False,
    cofactor=5,
    add_metadata=True
):

    adata = sc.read_h5ad(file_path)

    print(f"\nLoaded: {file_path}")
    print(adata)

    adata.obs["sample_id"] = adata.obs["sample_id"].str.replace(
        "_raw", "", regex=False
    )

    # =====================================================
    # metadata
    # =====================================================

    if add_metadata:

        if "sample_id" not in adata.obs.columns:
            raise ValueError("obs 中缺少 sample_id 列")

        sid = adata.obs["sample_id"].astype(str)

        def infer_group(x):

            if re.search(r"subtype_A", x):
                return "A"

            if re.search(r"-L", x):
                return "L"

            if re.search(r"subtype_V[0-9]+-G", x):
                return "G"

            return None

        adata.obs["Group"] = sid.apply(infer_group)

        print("\nGroup distribution:")
        print(adata.obs["Group"].value_counts())

        # =====================================================
        # split
        # =====================================================

        split_dash = adata.obs["sample_id"].str.split(
            "-",
            expand=True
        )

        adata.obs["part1"] = split_dash[0]
        adata.obs["part2"] = split_dash[1]
        adata.obs["part3"] = split_dash[2]

        # =====================================================
        # Group A
        # =====================================================

        mask_A = adata.obs["Group"] == "A"

        adata.obs.loc[mask_A, "Participant"] = (
            adata.obs.loc[mask_A, "part1"]
        )

        adata.obs.loc[mask_A, "Timepoint"] = (
            adata.obs.loc[mask_A, "part2"]
            .str.split("_")
            .str[0]
        )

        adata.obs.loc[mask_A, "Exp_id"] = (
            adata.obs.loc[mask_A, "part2"]
            .str.split("_")
            .str[1]
        )

        adata.obs.loc[mask_A, "Exp_position"] = (
            adata.obs.loc[mask_A, "part3"]
        )

        # =====================================================
        # Group G/L
        # =====================================================

        mask_GL = adata.obs["Group"].isin(["G", "L"])

        adata.obs.loc[mask_GL, "Timepoint"] = (
            adata.obs.loc[mask_GL, "part1"]
        )

        adata.obs.loc[mask_GL, "Participant"] = (
            adata.obs.loc[mask_GL, "part2"]
            .str.split("_")
            .str[0]
        )

        adata.obs.loc[mask_GL, "Exp_id"] = (
            adata.obs.loc[mask_GL, "part2"]
            .str.split("_")
            .str[1]
        )

        adata.obs.loc[mask_GL, "Exp_position"] = (
            adata.obs.loc[mask_GL, "part3"]
        )

        # =====================================================
        # cleanup
        # =====================================================

        adata.obs.drop(
            columns=["part1", "part2", "part3"],
            inplace=True
        )

        adata.obs["Participant"] = (
            adata.obs["Participant"]
            .str.replace("^subtype_", "", regex=True)
        )

        adata.obs["Timepoint"] = (
            adata.obs["Timepoint"]
            .str.replace("^subtype_", "", regex=True)
        )

        # =====================================================
        # timepoint mapping
        # =====================================================

        def map_timepoint(row):

            grp = row["Group"]
            tp = row["Timepoint"]

            # ===== Group G =====
            if grp == "G" and tp == "V1":
                return "V1Day0"

            elif grp == "G" and tp == "V2":
                return "V1Day1"

            # ===== Group L =====
            elif grp == "L" and tp == "V0":
                return "V1Day0"

            elif grp == "L" and tp == "V1":
                return "V1Day1"

            elif grp == "L" and tp == "V5":
                return "V2Day0"

            elif grp == "L" and tp == "V6":
                return "V2Day1"

            # ===== Group A =====
            elif grp == "A" and tp == "V0":
                return "V1Day0"

            elif grp == "A" and tp == "V1":
                return "V1Day1"

            elif grp == "A" and tp == "V3":
                return "V2Day0"

            elif grp == "A" and tp == "V4":
                return "V2Day1"

            elif grp == "A" and tp == "V6":
                return "V1Day0"

            elif grp == "A" and tp == "V7":
                return "V1Day1"

            elif grp == "A" and tp == "V8":
                return "V2Day0"

            elif grp == "A" and tp == "V9":
                return "V2Day1"

            return None

        adata.obs["Timepoint_V"] = adata.obs.apply(
            map_timepoint,
            axis=1
        )

        adata.obs["GT"] = (
            adata.obs["Group"].astype(str)
            + "_"
            + adata.obs["Timepoint_V"].astype(str)
        )

        print("Metadata parsing finished")

    # =====================================================
    # arcsinh
    # =====================================================

    if do_arcsinh:

        if "raw" not in adata.layers:
            adata.layers["raw"] = adata.X.copy()
            print("Raw layer saved")

        adata.X = np.arcsinh(adata.X / cofactor)

        print(f"arcsinh finished (cofactor={cofactor})")

    return adata


In [4]:
# =========================================================
# 绘图函数
# =========================================================

def plot_box_scatter(
    df,
    y,
    ylabel,
    save_dir,
    marker,
    ct
):

    df = df.dropna(subset=[y])

    if df.empty:
        return

    plt.figure(figsize=(6,5))

    sns.boxplot(
        data=df,
        x="Timepoint",
        y=y,
        order=timepoint_order,
        color="lightgray",
        showfliers=False
    )

    sns.stripplot(
        data=df,
        x="Timepoint",
        y=y,
        order=timepoint_order,
        hue="Participant",
        dodge=False,
        size=5,
        alpha=0.8,
        jitter=True
    )

    plt.yscale("symlog")

    plt.ylabel(ylabel)
    plt.xlabel("Timepoint")

    plt.title(f"{marker} ({ct})")

    plt.legend([], [], frameon=False)

    plt.tight_layout()

    plt.savefig(
        os.path.join(save_dir, f"{marker}_{ct}.png"),
        dpi=300
    )

    plt.close()

In [5]:
all_files = [
    f for f in os.listdir(input_dir)
    if f.endswith(".h5ad")
]


In [6]:
import numpy as np

def subsample_adata(adata, n=10000, seed=42):
    if adata.n_obs <= n:
        return adata

    np.random.seed(seed)
    idx = np.random.choice(adata.n_obs, n, replace=False)

    return adata[idx].copy()

# test3

In [ ]:
# =========================================================
# 主循环
# =========================================================

print(f"\nFound {len(all_files)} h5ad files")

for file in all_files:

    try:

        print("\n================================================")
        print(f"Processing: {file}")
        print("================================================")

        file_path = os.path.join(input_dir, file)
        sample_name = file.replace(".h5ad", "")

        adata = load_cytof_h5ad(file_path)
        adata.obs["celltype"] = sample_name

        # =====================================================
        # 输出目录
        # =====================================================

        outdir = os.path.join(output_root, sample_name)

        os.makedirs(outdir, exist_ok=True)

        dir1 = os.path.join(outdir, "1_threshold_distribution")
        dir2 = os.path.join(outdir, "2_positive_ratio")
        dir3 = os.path.join(outdir, "3_positive_median")
        dir4 = os.path.join(outdir, "4_positive_geomean")

        for d in [dir1, dir2, dir3, dir4]:
            os.makedirs(d, exist_ok=True)

        # =====================================================
        # celltypes（单一）
        # =====================================================

        celltypes = sorted(
            adata.obs["celltype"].dropna().unique()
        )

        participants = adata.obs["Participant"].unique()

        # =====================================================
        # baseline SD（V1Day0）
        # =====================================================

        baseline_idx = (
            adata.obs["Timepoint_V"] == "V1Day0"
        )

        baseline_sd = {}

        for marker in sub_markers:

            if marker not in adata.var_names:
                continue

            idx_marker = adata.var_names.get_loc(marker)

            data_vec = adata[baseline_idx, idx_marker].X
            data_vec = data_vec.toarray().flatten() if hasattr(data_vec, "toarray") else np.ravel(data_vec)

            baseline_sd[marker] = np.std(data_vec)

        # =====================================================
        # per-sample threshold
        # =====================================================

        marker_stats = {}

        for ct in celltypes:

            for marker in sub_markers:

                if marker not in adata.var_names:
                    continue

                idx_marker = adata.var_names.get_loc(marker)

                sd_base = baseline_sd.get(marker, np.nan)
                if np.isnan(sd_base):
                    continue

                for participant in participants:

                    mask = (
                        (adata.obs["Participant"] == participant)
                        &
                        (adata.obs["celltype"] == ct)
                    )

                    if mask.sum() == 0:
                        continue

                    data_vec = adata[mask, idx_marker].X
                    data_vec = data_vec.toarray().flatten() if hasattr(data_vec, "toarray") else np.ravel(data_vec)

                    median = np.median(data_vec)

                    threshold = median + 2 * sd_base

                    marker_stats[(participant, ct, marker)] = threshold

        # =====================================================
        # summary
        # =====================================================

        records = []

        for ct in celltypes:

            for marker in sub_markers:

                if marker not in adata.var_names:
                    continue

                idx_marker = adata.var_names.get_loc(marker)

                for participant in participants:

                    for tp in timepoint_order:

                        key = (participant, ct, marker)

                        if key not in marker_stats:
                            continue

                        threshold = marker_stats[key]

                        mask = (
                            (adata.obs["Participant"] == participant)
                            &
                            (adata.obs["Timepoint_V"] == tp)
                            &
                            (adata.obs["celltype"] == ct)
                        )

                        if mask.sum() == 0:
                            continue

                        data_vec = adata[mask, idx_marker].X
                        data_vec = data_vec.toarray().flatten() if hasattr(data_vec, "toarray") else np.ravel(data_vec)

                        pos = data_vec[data_vec > threshold]

                        records.append({
                            "Participant": participant,
                            "Timepoint": tp,
                            "Celltype": ct,
                            "Marker": marker,
                            "Positive_Ratio_%": len(pos)/len(data_vec)*100,
                            "Positive_Median": np.median(pos) if len(pos) > 0 else np.nan,
                            "Positive_Geomean": np.exp(np.mean(np.log(pos + 1e-9))) if len(pos) > 0 else np.nan
                        })

        marker_summary = pd.DataFrame(records)

        marker_summary.to_csv(os.path.join(outdir, "marker_summary.csv"), index=False)

        # =====================================================
        # threshold export
        # =====================================================

        threshold_df = pd.DataFrame([
            {
                "Participant": k[0],
                "Celltype": k[1],
                "Marker": k[2],
                "Threshold": v
            }
            for k, v in marker_stats.items()
        ])

        threshold_df.to_csv(os.path.join(outdir, "thresholds.csv"), index=False)

        # =====================================================
        # distribution plot（FIXED COLOR + LINE MATCH）
        # =====================================================

        import matplotlib.pyplot as plt

        cmap = plt.get_cmap("tab10").resampled(len(participants))

        for ct in celltypes:

            for marker in sub_markers:

                if marker not in adata.var_names:
                    continue

                idx_marker = adata.var_names.get_loc(marker)

                plt.figure(figsize=(6,5))

                for i, participant in enumerate(participants):

                    mask = (
                        (adata.obs["Participant"] == participant)
                        &
                        (adata.obs["celltype"] == ct)
                    )

                    if mask.sum() == 0:
                        continue

                    data_vec = adata[mask, idx_marker].X
                    data_vec = data_vec.toarray().flatten() if hasattr(data_vec, "toarray") else np.ravel(data_vec)

                    if np.std(data_vec) == 0:
                        continue

                    color = cmap(i)

                    sns.kdeplot(data_vec, color=color, alpha=0.3)

                    key = (participant, ct, marker)

                    if key in marker_stats:
                        plt.axvline(
                            marker_stats[key],
                            color=color,
                            linestyle="--",
                            linewidth=1
                        )

                plt.title(f"{marker} ({ct})")
                plt.xlabel("Expression")
                plt.ylabel("Density")
                plt.legend([], [], frameon=False)
                plt.tight_layout()

                plt.savefig(os.path.join(dir1, f"{marker}_{ct}.png"), dpi=300)
                plt.close()

        # =====================================================
        # box plots（不变）
        # =====================================================

        for ct in celltypes:
            for marker in sub_markers:

                df = marker_summary[
                    (marker_summary["Marker"] == marker)
                    &
                    (marker_summary["Celltype"] == ct)
                ]

                if df.empty:
                    continue

                plot_box_scatter(df, "Positive_Ratio_%", "Positive Ratio (%)", dir2, marker, ct)
                plot_box_scatter(df, "Positive_Median", "Median Expression", dir3, marker, ct)
                plot_box_scatter(df, "Positive_Geomean", "Geometric Mean", dir4, marker, ct)

        # =====================================================
        # statistics（FIXED SAFE ALIGN）
        # =====================================================

        results = []

        timepoint_pairs = [
            ("V1Day0", "V1Day1"),
            ("V2Day0", "V2Day1")
        ]

        for ct in celltypes:

            for marker in sub_markers:

                df = marker_summary[
                    (marker_summary["Celltype"] == ct)
                    &
                    (marker_summary["Marker"] == marker)
                ]

                if df.empty:
                    continue

                for tp1, tp2 in timepoint_pairs:

                    df1 = df[df["Timepoint"] == tp1]
                    df2 = df[df["Timepoint"] == tp2]

                    common = list(set(df1["Participant"]).intersection(df2["Participant"]))
                    if len(common) < 3:
                        continue

                    df1 = df1[df1["Participant"].isin(common)].set_index("Participant")
                    df2 = df2[df2["Participant"].isin(common)].set_index("Participant")

                    for metric in ["Positive_Ratio_%", "Positive_Median", "Positive_Geomean"]:

                        x = df1.loc[common, metric].values
                        y = df2.loc[common, metric].values

                        x = np.asarray(x, dtype=float)
                        y = np.asarray(y, dtype=float)

                        if len(x) < 3:
                            continue

                        if np.all(x == y):
                            continue

                        try:
                            stat, p = wilcoxon(x, y)
                        except:
                            continue

                        results.append({
                            "Celltype": ct,
                            "Marker": marker,
                            "Comparison": f"{tp1} vs {tp2}",
                            "Metric": metric,
                            "N": len(x),
                            "Median_1": np.median(x),
                            "Median_2": np.median(y),
                            "p_value": p
                        })

        results_df = pd.DataFrame(results)

        if not results_df.empty:
            results_df["p_adj"] = multipletests(results_df["p_value"], method="fdr_bh")[1]

        results_df.to_excel(os.path.join(outdir, "significant_markers_with_FDR.xlsx"), index=False)

        print(f"\n✓ Finished: {sample_name}")

    except Exception as e:
        print(f"\nERROR in {file}")
        print(e)

print("\n✅ ALL DONE")


Found 25 h5ad files

Processing: CD8TCM.h5ad

Loaded: /public/users/xueyupeng/VZV/cytof/subtype_combined/CD8TCM.h5ad
AnnData object with n_obs × n_vars = 6926190 × 52
    obs: 'sample_id'

Group distribution:
Group
A    3209966
L    2583068
G    1133156
Name: count, dtype: int64
Metadata parsing finished

✓ Finished: CD8TCM

Processing: NKdim.h5ad

Loaded: /public/users/xueyupeng/VZV/cytof/subtype_combined/NKdim.h5ad
AnnData object with n_obs × n_vars = 14753626 × 52
    obs: 'sample_id'

Group distribution:
Group
A    7494065
L    4522671
G    2736890
Name: count, dtype: int64
Metadata parsing finished

✓ Finished: NKdim

Processing: CD4TEMRA.h5ad

Loaded: /public/users/xueyupeng/VZV/cytof/subtype_combined/CD4TEMRA.h5ad
AnnData object with n_obs × n_vars = 838909 × 52
    obs: 'sample_id'

Group distribution:
Group
A    403076
L    285350
G    150483
Name: count, dtype: int64
Metadata parsing finished

✓ Finished: CD4TEMRA

Processing: CD8TEMRA.h5ad

Loaded: /public/users/xueyupeng/V